In [4]:
!nvidia-smi
!python --version

Sat Sep 26 23:58:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
from kaggle_secrets import UserSecretsClient
import os
import subprocess
from pathlib import Path

repo = "/kaggle/working/do-an-ptit-d22"

user_secrets = UserSecretsClient()
token = user_secrets.get_secret("GH_PAT")

askpass = Path("/kaggle/working/git-askpass.sh")
askpass.write_text(
    """#!/bin/sh
case "$1" in
  *Username*) echo "x-access-token" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
esac
"""
)
askpass.chmod(0o700)

env = os.environ.copy()
env["GITHUB_TOKEN"] = token
env["GIT_ASKPASS"] = str(askpass)
env["GIT_TERMINAL_PROMPT"] = "0"

subprocess.run(
    [
        "git",
        "clone",
        "--branch",
        "feat/m1-02-concurrency-sweep-baseline",
        "--single-branch",
        "https://github.com/theflyingkhui04/do-an-ptit-d22.git",
        repo,
    ],
    env=env,
    check=True,
)

askpass.unlink(missing_ok=True)

print("Clone OK")

Cloning into '/kaggle/working/do-an-ptit-d22'...


Clone OK


In [6]:
!git -C /kaggle/working/do-an-ptit-d22 rev-parse HEAD
!git -C /kaggle/working/do-an-ptit-d22 log -1 --oneline
!git -C /kaggle/working/do-an-ptit-d22 status --short

0e7a5a521d6f4c929d958a9eb4f65ead6cb33ece
0e7a5a5 (HEAD -> feat/m1-02-concurrency-sweep-baseline, origin/feat/m1-02-concurrency-sweep-baseline) feat(m1-02): add automatic concurrency sweep


In [7]:
!grep -n \
  -e "DEFAULT_CONCURRENCY_SWEEP" \
  -e "run_sweep" \
  -e "run_cell" \
  -e "run_repeat" \
  -e "CANONICAL_MIN_REPEATS" \
  /kaggle/working/do-an-ptit-d22/experiments/benchmark_harness.py

33:DEFAULT_CONCURRENCY_SWEEP = (1, 2, 4, 8, 16, 32)
34:CANONICAL_MIN_REPEATS = 5
462:async def run_repeat(
542:async def run_cell(
573:        repeat_code = await run_repeat(
599:async def run_sweep(
618:        cell_code = await run_cell(
648:    return list(DEFAULT_CONCURRENCY_SWEEP)
679:        canonical_sweep = list(DEFAULT_CONCURRENCY_SWEEP)
687:        if args.repeats < CANONICAL_MIN_REPEATS:
691:                f"M1 minimum {CANONICAL_MIN_REPEATS}."
705:            return await run_sweep(
772:        default=CANONICAL_MIN_REPEATS,


In [8]:
import torch
import importlib.metadata as m

print("torch:", torch.__version__)
print("torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("compute capability:", torch.cuda.get_device_capability(0))

for package in ["vllm", "aiohttp", "nvidia-ml-py"]:
    try:
        print(package, m.version(package))
    except m.PackageNotFoundError:
        print(package, "NOT INSTALLED")

torch: 2.13.0+cu130
torch CUDA: 13.0
CUDA available: True
GPU: Tesla T4
compute capability: (7, 5)
vllm 0.29.0
aiohttp 3.13.5
nvidia-ml-py 13.595.45


In [9]:
!cd /kaggle/working/do-an-ptit-d22 && \
python -m py_compile experiments/benchmark_harness.py

In [10]:
!cd /kaggle/working/do-an-ptit-d22 && \
python experiments/benchmark_harness.py --help

usage: benchmark_harness.py [-h] [--endpoint ENDPOINT] --model MODEL
                            [--prompt PROMPT] [--num-requests NUM_REQUESTS]
                            [--concurrency CONCURRENCY | --concurrency-sweep C [C ...]]
                            [--repeats REPEATS]
                            [--warmup-requests WARMUP_REQUESTS]
                            [--max-tokens MAX_TOKENS]
                            [--temperature TEMPERATURE] [--seed SEED]
                            [--lane {vanilla,static,adaptive}]
                            [--workload WORKLOAD]
                            [--experiment-id EXPERIMENT_ID]
                            [--timeout-s TIMEOUT_S] [--api-key API_KEY]
                            [--gpu-index GPU_INDEX]
                            [--vram-sample-interval-s VRAM_SAMPLE_INTERVAL_S]
                            [--raw-out RAW_OUT] [--summary-csv SUMMARY_CSV]
                            [--overwrite]

M1-02 vLLM benchmark harness with aut

In [11]:
import os
import subprocess

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"

log_file = open("/kaggle/working/vllm_server.log", "w")

server_process = subprocess.Popen(
    [
        "vllm",
        "serve",
        "Qwen/Qwen2.5-7B-Instruct-AWQ",
        "--host",
        "127.0.0.1",
        "--port",
        "8000",
    ],
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print("PID:", server_process.pid)

PID: 239


In [26]:
!tail -n 100 /kaggle/working/vllm_server.log

(APIServer pid=239) INFO 09-26 23:59:14 [api_utils.py:347] 
(APIServer pid=239) INFO 09-26 23:59:14 [api_utils.py:347]        █     █     █▄   ▄█
(APIServer pid=239) INFO 09-26 23:59:14 [api_utils.py:347]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.29.0
(APIServer pid=239) INFO 09-26 23:59:14 [api_utils.py:347]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-7B-Instruct-AWQ
(APIServer pid=239) INFO 09-26 23:59:14 [api_utils.py:347]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=239) INFO 09-26 23:59:14 [api_utils.py:347] 
(APIServer pid=239) INFO 09-26 23:59:14 [api_utils.py:286] non-default args: {'model_tag': 'Qwen/Qwen2.5-7B-Instruct-AWQ', 'host': '127.0.0.1', 'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ'}
(APIServer pid=239) INFO 09-26 23:59:28 [model.py:684] Resolved architecture: Qwen2ForCausalLM
(APIServer pid=239) INFO 09-26 23:59:28 [model.py:2021] Using max model len 32768
(APIServer pid=239) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable hig

In [27]:
!curl -sS -o /dev/null -w "%{http_code}\n" \
  http://127.0.0.1:8000/health

200


In [28]:
!curl -sS http://127.0.0.1:8000/v1/models | python -m json.tool

{
    "object": "list",
    "data": [
        {
            "id": "Qwen/Qwen2.5-7B-Instruct-AWQ",
            "object": "model",
            "created": 1790468168,
            "owned_by": "vllm",
            "root": "Qwen/Qwen2.5-7B-Instruct-AWQ",
            "parent": null,
            "max_model_len": 32768,
            "permission": [
                {
                    "id": "modelperm-9e1e26bb6bfd8174",
                    "object": "model_permission",
                    "created": 1790468168,
                    "allow_create_engine": false,
                    "allow_sampling": true,
                    "allow_logprobs": true,
                    "allow_search_indices": false,
                    "allow_view": true,
                    "allow_fine_tuning": false,
                    "organization": "*",
                    "group": null,
                    "is_blocking": false
                }
            ]
        }
    ]
}


In [29]:
%%bash

curl -sS http://127.0.0.1:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "Qwen/Qwen2.5-7B-Instruct-AWQ",
    "messages": [
      {
        "role": "user",
        "content": "Trả lời đúng một từ: thủ đô của Việt Nam là gì?"
      }
    ],
    "temperature": 0,
    "max_tokens": 16
  }' | python -m json.tool

{
    "id": "chatcmpl-90b01abab2286e23",
    "object": "chat.completion",
    "created": 1790468171,
    "model": "Qwen/Qwen2.5-7B-Instruct-AWQ",
    "choices": [
        {
            "index": 0,
            "message": {
                "role": "assistant",
                "content": "H\u00e0 N\u1ed9i",
                "refusal": null,
                "annotations": null,
                "audio": null,
                "function_call": null,
                "reasoning": null
            },
            "logprobs": null,
            "finish_reason": "stop",
            "stop_reason": null,
            "token_ids": null,
            "routed_experts": null
        }
    ],
    "service_tier": null,
    "system_fingerprint": "vllm-0.29.0-3e2a96bd",
    "usage": {
        "prompt_tokens": 44,
        "total_tokens": 48,
        "completion_tokens": 4,
        "prompt_tokens_details": null,
        "completion_tokens_details": null
    },
    "prompt_logprobs": null,
    "prompt_token_ids": n

# **TEST A — một cell, 2 repeats**

In [30]:
!rm -f /kaggle/working/do-an-ptit-d22/experiments/runs/sweep_smoke_c4.jsonl

!cd /kaggle/working/do-an-ptit-d22 && \
python experiments/benchmark_harness.py \
  --model Qwen/Qwen2.5-7B-Instruct-AWQ \
  --lane vanilla \
  --workload smoke \
  --concurrency 4 \
  --num-requests 8 \
  --repeats 2 \
  --warmup-requests 3 \
  --max-tokens 16 \
  --experiment-id exp-m1-02-sweep-smoke-c4 \
  --raw-out experiments/runs/sweep_smoke_c4.jsonl \
  --overwrite


===== ENVIRONMENT CONTRACT PASS =====
profile              : m1-t4-cu130-v1
gpu                  : Tesla T4
vllm_version         : 0.29.0
torch_version        : 2.13.0+cu130
torch_cuda_version   : 13.0

===== NON-CANONICAL CONCURRENCY SET =====
selected             : [4]
canonical            : [1, 2, 4, 8, 16, 32]
Do not use this run as complete M1 sweep evidence.

===== NON-CANONICAL SMOKE MODE =====
repeats=2 is below canonical M1 minimum 5.
Do not use this run as scientific M1 benchmark evidence.

===== M1-02 CONCURRENCY SWEEP =====
experiment_id        : exp-m1-02-sweep-smoke-c4
lane                 : vanilla
workload             : smoke
concurrencies        : [4]
repeats_per_cell     : 2
requests_per_repeat  : 8

START CELL
experiment_id : exp-m1-02-sweep-smoke-c4
cell_id       : vanilla__smoke__c4
concurrency   : 4
repeats       : 2

===== M1-02 CELL WARM-UP =====
policy               : option-b-concurrency-matched
experiment_id        : exp-m1-02-sweep-smoke-c4
cell_id         

# **TEST B — verify hierarchy + raw append**

In [31]:
import json
from collections import Counter

path = (
    "/kaggle/working/do-an-ptit-d22/"
    "experiments/runs/sweep_smoke_c4.jsonl"
)

with open(path, encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

print("rows:", len(rows))
print("experiments:", {r["experiment_id"] for r in rows})
print("cells:", {r["cell_id"] for r in rows})
print("repeats:", {r["repeat_id"] for r in rows})
print("phases:", {r["phase"] for r in rows})
print("rows/repeat:", Counter(r["repeat_id"] for r in rows))

assert len(rows) == 16

assert {r["experiment_id"] for r in rows} == {
    "exp-m1-02-sweep-smoke-c4"
}

assert {r["cell_id"] for r in rows} == {
    "vanilla__smoke__c4"
}

assert {r["repeat_id"] for r in rows} == {
    "r001",
    "r002",
}

assert {r["phase"] for r in rows} == {
    "measured"
}

assert Counter(r["repeat_id"] for r in rows) == {
    "r001": 8,
    "r002": 8,
}

assert len({r["request_id"] for r in rows}) == 16

print("SINGLE-CELL REPEAT ORCHESTRATION: PASS")

rows: 16
experiments: {'exp-m1-02-sweep-smoke-c4'}
cells: {'vanilla__smoke__c4'}
repeats: {'r001', 'r002'}
phases: {'measured'}
rows/repeat: Counter({'r001': 8, 'r002': 8})
SINGLE-CELL REPEAT ORCHESTRATION: PASS


# **TEST C — mini sweep 1 → 2 → 4**

In [32]:
!rm -f /kaggle/working/do-an-ptit-d22/experiments/runs/mini_sweep.jsonl

!cd /kaggle/working/do-an-ptit-d22 && \
python experiments/benchmark_harness.py \
  --model Qwen/Qwen2.5-7B-Instruct-AWQ \
  --lane vanilla \
  --workload smoke \
  --concurrency-sweep 1 2 4 \
  --num-requests 8 \
  --repeats 2 \
  --warmup-requests 3 \
  --max-tokens 16 \
  --experiment-id exp-m1-02-mini-sweep \
  --raw-out experiments/runs/mini_sweep.jsonl \
  --overwrite


===== ENVIRONMENT CONTRACT PASS =====
profile              : m1-t4-cu130-v1
gpu                  : Tesla T4
vllm_version         : 0.29.0
torch_version        : 2.13.0+cu130
torch_cuda_version   : 13.0

===== NON-CANONICAL CONCURRENCY SET =====
selected             : [1, 2, 4]
canonical            : [1, 2, 4, 8, 16, 32]
Do not use this run as complete M1 sweep evidence.

===== NON-CANONICAL SMOKE MODE =====
repeats=2 is below canonical M1 minimum 5.
Do not use this run as scientific M1 benchmark evidence.

===== M1-02 CONCURRENCY SWEEP =====
experiment_id        : exp-m1-02-mini-sweep
lane                 : vanilla
workload             : smoke
concurrencies        : [1, 2, 4]
repeats_per_cell     : 2
requests_per_repeat  : 8

START CELL
experiment_id : exp-m1-02-mini-sweep
cell_id       : vanilla__smoke__c1
concurrency   : 1
repeats       : 2

===== M1-02 CELL WARM-UP =====
policy               : option-b-concurrency-matched
experiment_id        : exp-m1-02-mini-sweep
cell_id         

# **TEST D — validate mini sweep raw**

In [33]:
import json
from collections import Counter

path = (
    "/kaggle/working/do-an-ptit-d22/"
    "experiments/runs/mini_sweep.jsonl"
)

with open(path, encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

print("rows:", len(rows))

print("experiment IDs:")
print({r["experiment_id"] for r in rows})

print("cell IDs:")
print(sorted({r["cell_id"] for r in rows}))

print("repeat IDs:")
print(sorted({r["repeat_id"] for r in rows}))

counts = Counter(
    (r["cell_id"], r["repeat_id"])
    for r in rows
)

for key, value in sorted(counts.items()):
    print(key, value)

assert len(rows) == 3 * 2 * 8

assert {r["experiment_id"] for r in rows} == {
    "exp-m1-02-mini-sweep"
}

assert {r["cell_id"] for r in rows} == {
    "vanilla__smoke__c1",
    "vanilla__smoke__c2",
    "vanilla__smoke__c4",
}

assert {r["repeat_id"] for r in rows} == {
    "r001",
    "r002",
}

assert {r["phase"] for r in rows} == {
    "measured"
}

assert all(value == 8 for value in counts.values())

print("MINI CONCURRENCY SWEEP: PASS")

rows: 48
experiment IDs:
{'exp-m1-02-mini-sweep'}
cell IDs:
['vanilla__smoke__c1', 'vanilla__smoke__c2', 'vanilla__smoke__c4']
repeat IDs:
['r001', 'r002']
('vanilla__smoke__c1', 'r001') 8
('vanilla__smoke__c1', 'r002') 8
('vanilla__smoke__c2', 'r001') 8
('vanilla__smoke__c2', 'r002') 8
('vanilla__smoke__c4', 'r001') 8
('vanilla__smoke__c4', 'r002') 8
MINI CONCURRENCY SWEEP: PASS


# **full automatic sweep smoke**

In [34]:
!rm -f /kaggle/working/do-an-ptit-d22/experiments/runs/full_sweep_smoke.jsonl

!cd /kaggle/working/do-an-ptit-d22 && \
python experiments/benchmark_harness.py \
  --model Qwen/Qwen2.5-7B-Instruct-AWQ \
  --lane vanilla \
  --workload smoke \
  --num-requests 32 \
  --repeats 1 \
  --warmup-requests 3 \
  --max-tokens 16 \
  --experiment-id exp-m1-02-full-sweep-smoke \
  --raw-out experiments/runs/full_sweep_smoke.jsonl \
  --overwrite


===== ENVIRONMENT CONTRACT PASS =====
profile              : m1-t4-cu130-v1
gpu                  : Tesla T4
vllm_version         : 0.29.0
torch_version        : 2.13.0+cu130
torch_cuda_version   : 13.0

===== NON-CANONICAL SMOKE MODE =====
repeats=1 is below canonical M1 minimum 5.
Do not use this run as scientific M1 benchmark evidence.

===== M1-02 CONCURRENCY SWEEP =====
experiment_id        : exp-m1-02-full-sweep-smoke
lane                 : vanilla
workload             : smoke
concurrencies        : [1, 2, 4, 8, 16, 32]
repeats_per_cell     : 1
requests_per_repeat  : 32

START CELL
experiment_id : exp-m1-02-full-sweep-smoke
cell_id       : vanilla__smoke__c1
concurrency   : 1
repeats       : 1

===== M1-02 CELL WARM-UP =====
policy               : option-b-concurrency-matched
experiment_id        : exp-m1-02-full-sweep-smoke
cell_id              : vanilla__smoke__c1
requested_minimum    : 3
warmup_requests      : 3
warmup_concurrency   : 1
measured_concurrency : 1

===== WARM-UP 

In [35]:
import json
from collections import Counter

path = (
    "/kaggle/working/do-an-ptit-d22/"
    "experiments/runs/full_sweep_smoke.jsonl"
)

with open(path, encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

expected_cells = {
    "vanilla__smoke__c1",
    "vanilla__smoke__c2",
    "vanilla__smoke__c4",
    "vanilla__smoke__c8",
    "vanilla__smoke__c16",
    "vanilla__smoke__c32",
}

print("rows:", len(rows))
print("cells:", sorted({r["cell_id"] for r in rows}))

assert len(rows) == 192
assert {r["cell_id"] for r in rows} == expected_cells
assert {r["repeat_id"] for r in rows} == {"r001"}
assert {r["phase"] for r in rows} == {"measured"}

counts = Counter(r["cell_id"] for r in rows)

for cell in sorted(counts):
    print(cell, counts[cell])
    assert counts[cell] == 32

print("FULL 1→32 SWEEP SMOKE: PASS")

rows: 192
cells: ['vanilla__smoke__c1', 'vanilla__smoke__c16', 'vanilla__smoke__c2', 'vanilla__smoke__c32', 'vanilla__smoke__c4', 'vanilla__smoke__c8']
vanilla__smoke__c1 32
vanilla__smoke__c16 32
vanilla__smoke__c2 32
vanilla__smoke__c32 32
vanilla__smoke__c4 32
vanilla__smoke__c8 32
FULL 1→32 SWEEP SMOKE: PASS


# **chạy canonical 5-repeat sweep**

In [36]:
!rm -f /kaggle/working/do-an-ptit-d22/experiments/runs/vanilla_W1_sweep.jsonl

!cd /kaggle/working/do-an-ptit-d22 && \
python experiments/benchmark_harness.py \
  --model Qwen/Qwen2.5-7B-Instruct-AWQ \
  --lane vanilla \
  --workload W1 \
  --num-requests 32 \
  --repeats 5 \
  --warmup-requests 3 \
  --max-tokens 32 \
  --experiment-id exp-m1-02-vanilla-W1-001 \
  --raw-out experiments/runs/vanilla_W1_sweep.jsonl \
  --overwrite


===== ENVIRONMENT CONTRACT PASS =====
profile              : m1-t4-cu130-v1
gpu                  : Tesla T4
vllm_version         : 0.29.0
torch_version        : 2.13.0+cu130
torch_cuda_version   : 13.0

===== M1-02 CONCURRENCY SWEEP =====
experiment_id        : exp-m1-02-vanilla-W1-001
lane                 : vanilla
workload             : W1
concurrencies        : [1, 2, 4, 8, 16, 32]
repeats_per_cell     : 5
requests_per_repeat  : 32

START CELL
experiment_id : exp-m1-02-vanilla-W1-001
cell_id       : vanilla__W1__c1
concurrency   : 1
repeats       : 5

===== M1-02 CELL WARM-UP =====
policy               : option-b-concurrency-matched
experiment_id        : exp-m1-02-vanilla-W1-001
cell_id              : vanilla__W1__c1
requested_minimum    : 3
warmup_requests      : 3
warmup_concurrency   : 1
measured_concurrency : 1

===== WARM-UP PASS =====
cell_id              : vanilla__W1__c1
requests             : 3
successes            : 3
concurrency_limit    : 1

===== M1-02 REPEAT RESULT =